In [27]:
import json
import pandas as pd
import requests
from pathlib import Path

data_dir = Path('data')
if not data_dir.is_dir() and (Path('..') / 'data').is_dir():
    data_dir = Path('..') / 'data'
data_dir.mkdir(parents=True, exist_ok=True)

house = "https://live-data.jifo.co/3cba2007-e2ef-4450-b869-8d10da1d0c77"
senate = "https://live-data.jifo.co/eb6a9fd7-e81c-4d19-ba9f-7e408513d75e"
senate_map = "https://live-data.jifo.co/6911ecb8-c744-4464-a182-70568e288364"
house_map = "https://live-data.jifo.co/d6524bc4-7a1b-4c9a-93e4-da87f529c097"


In [28]:
def get_data(url):
    r = requests.get(url)
    return r.json()

raw_data = get_data(house)

In [29]:
chance_of_winning = raw_data['data'][0]

headers = chance_of_winning[0]
data_rows = chance_of_winning[1:]

df = pd.DataFrame(data_rows, columns=headers)

def CorrectDates(current_year):
    corrected_dates = []

    for i, row in df.iterrows():
        # Parse the month and day, then set it to the current_year
        # Using a format string for robustness
        dt = pd.to_datetime(row['Date'] + f' {current_year}', format='%b %d %Y')

        # If this date is before the previous one, it means a year has rolled over
        # and the year needs to be incremented to maintain chronological order.
        if i > 0 and dt < corrected_dates[-1]:
            current_year += 1
            dt = pd.to_datetime(row['Date'] + f' {current_year}', format='%b %d %Y')

        corrected_dates.append(dt)
    return corrected_dates

df['Full_Date'] = CorrectDates(2025)

df.to_csv(data_dir / 'house_chance_of_winning.csv', index=False)



In [30]:
current_projection = raw_data['data'][2]

headers = current_projection[0]
data_rows = [current_projection[- 1]]

df = pd.DataFrame(data_rows, columns=headers)
df.to_csv(data_dir / 'house_current_projection.csv', index=False)

In [31]:
raw_data = get_data(senate)

chance_of_winning = raw_data['data'][0]

headers = chance_of_winning[0]
data_rows = chance_of_winning[1:]

df = pd.DataFrame(data_rows, columns=headers)

def CorrectDates(current_year):
    corrected_dates = []

    for i, row in df.iterrows():
        # Parse the month and day, then set it to the current_year
        # Using a format string for robustness
        dt = pd.to_datetime(row['Date'] + f' {current_year}', format='%b %d %Y')

        # If this date is before the previous one, it means a year has rolled over
        # and the year needs to be incremented to maintain chronological order.
        if i > 0 and dt < corrected_dates[-1]:
            current_year += 1
            dt = pd.to_datetime(row['Date'] + f' {current_year}', format='%b %d %Y')

        corrected_dates.append(dt)
    return corrected_dates

df['Full_Date'] = CorrectDates(2025)

df.to_csv(data_dir / 'senate_chance_of_winning.csv', index=False)

current_projection = raw_data['data'][1]

headers = current_projection[0]
data_rows = [current_projection[- 1]]

df = pd.DataFrame(data_rows, columns=headers)
df.to_csv(data_dir / 'senate_current_projection.csv', index=False)

In [ ]:
raw_data = get_data(senate_map)

map = raw_data['data'][10]

headers = map[0]
data_rows = map[1:]

df = pd.DataFrame(data_rows, columns=headers)
df.to_csv(data_dir / 'senate_map.csv', index=False)

In [35]:
raw_data = get_data(house_map)
map = raw_data['data'][1]
headers = map[3]
data_rows = map[4:]
df = pd.DataFrame(data_rows, columns=headers)
parts = df["Race"].str.extract(r"^([A-Z]{2})\s*-\s*(\d+)$")
df["Race_clean"] = parts[0] + parts[1].str.zfill(2)
df.to_csv(data_dir / 'house_map.csv', index=False)